In [1]:
from utils import create_new_dataset_with_ipcw_weights, create_surv_data, train_test_split_into_df, get_Nbi, ipc_weighted_mse, inf_JK_bagged_variance
from lifelines import KaplanMeierFitter, WeibullAFTFitter
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

### create dataset

In [2]:
n = 1000
seed = 42

data = create_surv_data(shape_weibull=1,   # constant hazard
                        scale_weibull_base=10000, 
                        rate_censoring=0.01,  
                        b_bloodp=-0.405, 
                        b_diab=-0.4, 
                        b_age=-0.05, 
                        b_bmi=-0.01, 
                        b_kreat=-0.2,
                        n=n, 
                        seed=seed)

Data shape: (1000, 7)
34.0 % of the data has an event


### stratified train and test split

In [3]:
X = data[['bmi', 'blood_pressure', 'kreatinkinase', 'diabetes', 'age']]
y = Surv.from_arrays(event=data['event'], time=data['t'])

df_train, df_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)
df_train, df_test = train_test_split_into_df(df_train=df_train, df_test=df_test, y_train=y_train, y_test=y_test)

In [4]:
## Überprüfung der Verteilungen Train und Test Data

# fig, axes = plt.subplots(1, 2, figsize=(6, 4), sharey=True)
# sns.boxplot(ax=axes[0], x='event', y='time', data=df_train)
# axes[0].set_title('Training Data - Time by Event')
# axes[0].set_xlabel('Event')
# axes[0].set_ylabel('Time')
# sns.boxplot(ax=axes[1], x='event', y='time', data=df_test)
# axes[1].set_title('Test Data - Time by Event')
# axes[1].set_xlabel('Event')
# axes[1].set_ylabel('Time')
# y_min = min(df_train['time'].min(), df_test['time'].min())
# y_max = max(df_train['time'].max(), df_test['time'].max())
# axes[0].set_ylim(y_min, y_max)
# axes[1].set_ylim(y_min, y_max)
# plt.tight_layout()
# plt.show()

# train_event_counts = df_train['event'].value_counts(normalize=True)
# test_event_counts = df_test['event'].value_counts(normalize=True)
# print("Relative Häufigkeiten der Events - Trainingsdaten:")
# print(train_event_counts)
# print("\nRelative Häufigkeiten der Events - Testdaten:")
# print(test_event_counts)

# strg k str u  -  kommentar
# strg k str c  -  kommentar entfernen

### cut data at tau // ipcw weights


In [5]:
tau = 12

kmf = KaplanMeierFitter()
kmf.fit(df_train['time'], event_observed=1-df_train['event'])
df_train = create_new_dataset_with_ipcw_weights(data=df_train,t=tau, kmf=kmf)
df_test = create_new_dataset_with_ipcw_weights(data=df_test,t=tau, kmf=kmf)

# portion of censored data after cutpoint
portions_at_cutpoint = df_train['survived'].value_counts(normalize=True)
censored_at_t = portions_at_cutpoint[999]
print(f'train: censored after cut at tau={tau}: {censored_at_t*100:.2f} %')

portions_at_cutpoint = df_test['survived'].value_counts(normalize=True)
censored_at_t = portions_at_cutpoint[999]
print(f'test: censored after cut at tau={tau}: {censored_at_t*100:.2f} %')

print('\n')
print(df_train[df_train['survived']==0].sample(2)[['time', 'event','weights_ipcw','survived', ]])
print('\n')
print(df_train[df_train['survived']==1].sample(2)[['time', 'event','weights_ipcw','survived', ]])
print('\n')
print(df_train[df_train['survived']==999].sample(2)[['time', 'event','weights_ipcw','survived', ]])
print('\n')
print(df_train.columns)

### Wieso ist die Summe der ipcw_Gewichte gleich n  ??

train: censored after cut at tau=12: 12.43 %
test: censored after cut at tau=12: 9.00 %


          time  event  weights_ipcw  survived
392   7.026907   True      0.001553         0
401  11.014874   True      0.001625         0


          time  event  weights_ipcw  survived
635  17.695602  False      0.001639         1
647  89.601740  False      0.001639         1


         time  event  weights_ipcw  survived
192  0.838980  False           0.0       999
502  6.358749  False           0.0       999


Index(['bmi', 'blood_pressure', 'kreatinkinase', 'diabetes', 'age', 'event',
       'time', 'survived', 'weights_ipcw'],
      dtype='object')


### Weibull Modell fitten

In [ ]:
aft = WeibullAFTFitter()
aft.fit(df_train.drop(['weights_ipcw', 'survived'], axis=1), duration_col='time', event_col='event')
#aft.print_summary()

<lifelines.WeibullAFTFitter: fitted with 700 total observations, 462 right-censored observations>

### Weibull Modell Evaluation

In [7]:
# IPCW-MSE berechnen
y_pred = aft.predict_survival_function(df_test.drop(['weights_ipcw', 'survived','time','event'], axis=1), times = tau).iloc[0].tolist()
weibull_mse = ipc_weighted_mse(y_true=df_test['survived'].values, y_pred=y_pred, sample_weight=df_test['weights_ipcw'])
print(f'weibull_ipcw_mse auf test data: {round(weibull_mse,4)}')


# IPCW-C-Index berechnen
tied_risk_scores = aft.predict_expectation(df_test)
ci_ipcw, concordant, discordant, tied_risk, tied_time = concordance_index_ipcw(
    y_train, y_test, -tied_risk_scores )
print("IPCW-C-Index:", round(ci_ipcw,4))


weibull_ipcw_mse auf test data: 0.0679
IPCW-C-Index: 0.6921


In [8]:
X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})
y_pred_X_erwartung_weibull = aft.predict_survival_function(X_erwartung, times = tau).iloc[0].tolist()

print(f'Prediction für X_erwartung zum überleben am Zeitpunkt tau={tau}: {y_pred_X_erwartung_weibull[0]:.4f}')

Prediction für X_erwartung zum überleben am Zeitpunkt tau=12: 0.9498


### Random Forest Modell

In [9]:
X_train = df_train.drop(['time', 'event', 'weights_ipcw', 'survived'], axis=1)
y_train = df_train['survived']
bootstrap_weights = df_train['weights_ipcw']

In [10]:
# # HYperparameter Tuning
# param_dist = {
#     'n_estimators': [int(x) for x in np.linspace(start=X_train.shape[0], stop=X_train.shape[0]*10, num=10)],
#     'max_features': [ 'sqrt', 'log2'],
#     'max_depth': [int(x) for x in np.linspace(1,10 , num=10)] + [None],
#     'min_samples_split':[int(x) for x in np.linspace(1,10 , num=10)],
#     'bootstrap': [True]
# }

# rf = RandomForestClassifier(random_state=seed)
# rf_random = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=100, cv=5, verbose=3, random_state=seed, n_jobs=-1)
# rf_random.fit(X_train, y_train, sample_weight=bootstrap_weights)

# print(rf_random.best_params_)
# best_rf = rf_random.best_estimator_

# y_pred = best_rf.predict_proba(df_test.drop(['weights_ipcw', 'survived','time','event'], axis=1))[:,1]
# best_rf_mse = ipc_weighted_mse(y_true=df_test['survived'].values, y_pred=y_pred, sample_weight=df_test['weights_ipcw'])

# print(f'best_rf_ipcw_mse auf test data: {round(best_rf_mse,4)}')

In [11]:
params = {
    'n_estimators': X_train.shape[0]*2,
    'max_depth': 3,
    'min_samples_split': 5,
    'max_features': 'log2',
    'random_state': seed,
    'bootstrap': True
}

clf = RandomForestClassifier(**params)
clf.fit(X_train, y_train, sample_weight=bootstrap_weights)

y_pred = clf.predict_proba(df_test.drop(['weights_ipcw', 'survived','time','event'], axis=1))[:,1]
rf_mse = ipc_weighted_mse(y_true=df_test['survived'].values, y_pred=y_pred, sample_weight=df_test['weights_ipcw'])
print(f'rf_ipcw_mse auf test data: {round(rf_mse,4)}')

rf_ipcw_mse auf test data: 0.0688


#### Infinetesimal Jackknife

In [12]:
y_pred_X_erwartung = clf.predict_proba(X_erwartung)[:,1]
print(f'Prediction für X_erwartung zum überleben am Zeitpunkt tau={tau}: {y_pred_X_erwartung[0]:.4f}')

nbi = get_Nbi(clf.estimators_samples_)
tnb = np.array([tree.predict_proba(X_erwartung.values)[:, 1][0] for tree in clf.estimators_]) 

biased_var_estimate, bias_correction = inf_JK_bagged_variance(N_bi=nbi, T_N_b=tnb,weights=bootstrap_weights)
unbiased_var_estimate = biased_var_estimate - bias_correction
print(f'Unbiased std_ijk Estimate: {np.sqrt(unbiased_var_estimate):.4f}')

rf_std = np.std(tnb)
print(f'rf_std: {rf_std:.4f}')

Prediction für X_erwartung zum überleben am Zeitpunkt tau=12: 0.9474
Unbiased std_ijk Estimate: 0.0116
rf_std: 0.0301
